### Application of Embedding:

We have Embedding from Two different methods. <br>
Now, we will use these embedding and note the better one by using some evaluation metrics.

In [ ]:
import pandas as pd
from typing import List
import numpy as np

In [ ]:
Dataset = pd.read_parquet("../Dataset/Clean/Dataset_with_embeddings.parquet")
Embedding = pd.read_parquet("../Dataset/Clean/Embeddings.parquet")
print(Dataset.columns)
print(Dataset.shape)

print(Embedding.columns)
print(Embedding.shape)

## **Clustering Similarity**:
Clump together similar article based on important numerical features and Embedding of each article. <br>
We will use HDBSCAN for clustering. <br>

For better analysis, we will try cluster based on Heuristic counts + embeddings.

> Due to lots of outliers, we use Robust scalar -> X = (x-median)/IQR

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler
from typing import Literal, List, Optional
import joblib
from sklearn.decomposition import PCA
import umap
import hdbscan
from sklearn.metrics import silhouette_score

In [ ]:
Heuristic_cols = ['char_count', 'sentence_count', 'word_count', 'unique_word_count', 
    'lexical_diversity', 'hapax_ratio',
    'stopword_ratio', 
    'noun_count', 'verb_count', 'adj_count', 'adv_count', 'pronoun_count',
    'person_count', 'org_count', 'gpe_count', 'event_count', 'unique_entity_count',
    'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog',
    'word_ratio', 'char_ratio', 'sent_ratio',
]

In [ ]:
# Step-1: Create Clusterable matrix from Input
def prepare_clustering_matrix(df:pd.DataFrame, heuristic_cols:List[str], embedding_col:str='Embedding',
    embedding_weight:float=1.0, heuristic_weight:float=0.5, scaling_method: Optional[Literal["robust", "log-robust"]] = "robust",
    apply_reduction: bool = False, reduction_factor: Optional[float] = 2.0
) -> np.ndarray:
    """
    Prepare clustering matrix - Initial matrix to create Clustering

    Args:
        df (pd.DataFrame): Input dataset
        heuristic_cols (List[str]): List of all heuristic features
        embedding_col (str, optional): Column name for embedding. Defaults to 'Embedding'.
        embedding_weight (float, optional): Scaling factor for embeddings. Defaults to 1.0.
        heuristic_weight (float, optional): Scaling factor for heuristic features. Defaults to 0.5.
        scaling_method (Optional[Literal[&quot;robust&quot;, &quot;log, optional): Feature scaling method. Defaults to "robust".
        apply_reduction (bool, optional): apply dimensionality reduction to embedding?. Default - False.
        reduction_factor (Optional[float], optional): Preceding PCA factor -> PCA will reduce embedding down to d_heuristic * reduction_factor. Defaults - 2.

    Returns:
        np.ndarray: Clustering matrix
    """
    
    # -- Heuristic features --
    data = df[heuristic_cols]
    
    if scaling_method == "robust":
        heuristic_matrix = RobustScaler().fit_transform(data)
    elif scaling_method == "log-robust":
        heuristic_matrix = RobustScaler(quantile_range=(25, 75)).fit_transform(np.log1p(data))
    else:
        raise ValueError(f"Unknown scaling method: {scaling_method}")
     
    heuristic_matrix = heuristic_weight*heuristic_matrix

    # -- Embeddings --
    embedding_matrix = np.vstack(df[embedding_col].values) # Stack embeddings
    
    # --- Optional: Dimensionality reduction on embeddings ---
    if apply_reduction:
        target_dim = min(embedding_matrix.shape[1], int(reduction_factor*heuristic_matrix.shape[1]))  # Match heuristic feature dimensions
        if target_dim < 2:
            raise ValueError("Cannot reduce embeddings to less than 2 dimensions.")
        
        print(f"Reducing embeddings from {embedding_matrix.shape[1]} to {target_dim}")
        
        # # Prepend PCA to speed up
        pca = PCA(n_components=target_dim)
        embedding_matrix = pca.fit_transform(embedding_matrix)
        
        # Umap
        # reducer = umap.UMAP(n_components=target_dim, n_neighbors=5, min_dist=0.01, metric='cosine')
        # embedding_matrix = reducer.fit_transform(embedding_matrix)
        
        # Save the reducer for future use (e.g., transforming new data)
        joblib.dump(pca, '../Models/reducer_pca.joblib')
        # joblib.dump(reducer, '../Models/reducer_umap.joblib')

    embedding_matrix = embedding_weight * StandardScaler().fit_transform(embedding_matrix)
    # --- Final clustering matrix ---
    clustering_matrix = np.hstack([heuristic_matrix, embedding_matrix])

    return clustering_matrix
    

##### HDBSCAN - Hierarchical Density-Based Spatial Clustering of Applications with Noise
It is an unsupervised ml algorithm which Performs `DBSCAN` over varying epsilon values and <br>
integrates the result to find a clustering that gives the best stability over epsilon. <br>
This allows **`HDBSCAN`** to find clusters of varying densities (unlike DBSCAN), and be more robust to parameter selection. 

##### MiniBatch K-Means
It is a variant of K-Means that uses mini-batches to reduce the computation time while still trying to optimize the same objective function. <br>
It is particularly useful for large datasets, as it can converge faster than standard K-Means by using small random samples of the data to update the cluster centers. <br>
However, it may not always find the optimal solution and can be sensitive to the choice of batch size and initialization.

In [ ]:
# Step-2: Create Clusters
def Cluster_articles_hdbscan(clustering_matrix):
    """
    Create Clusters and Label each Input Article
    
    Args:
        clustering_matrix (_type_): _description_
        min_cluster_size (int, optional): _description_. Defaults to 100.

    Returns:
        _type_: _description_
    """
    clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5,
        metric='euclidean', cluster_selection_method='eom'
    )
    labels = clusterer.fit_predict(clustering_matrix)
    return clusterer, labels

def evaluate_hdbscan(clustering_matrix:np.ndarray, labels:np.ndarray):
        score = silhouette_score(clustering_matrix, labels, metric='euclidean')
        return score


> HDBSCAN did not give any meaningful results. So, we will switch over to K-Means.

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# Step-3: Find Optimal Clusters
class SemanticClustering:
    def __init__(self, n_clusters=10, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.model = None
        
    def fit(self, clustering_matrix:np.ndarray, clusters:int=None):
        if clusters is not None:
            self.n_clusters = clusters
        self.model = MiniBatchKMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init='auto')
        labels = self.model.fit_predict(clustering_matrix)
        
        return labels
    
    def evaluate(self, clustering_matrix:np.ndarray, labels:np.ndarray):
        score = silhouette_score(clustering_matrix, labels, metric='euclidean')
        return score
    
    def save(self, path:str):
        joblib.dump(self.model, path)
        
    def laod(self, path:str):
        self.model = joblib.load(path)

#### Results: <br>
We used our Embedding to create Cluster of Article which can be used to categorization article by Zero-shot learning. <br>

After clustering + zero-shot labeling, you now have:
- Semantic groups of articles
- Each group has:
  - a dominant topic
  - similar writing style
  - similar complexity
- Articles are indexed into semantic regions

This gives you ***latent*** structure.

At Inference time, we can use this in the following ways:
1. Topic-Aware summarization -> Summarize based on dominant topic which improve `ROUGE` Metric by a lot.
2. Cold-start classifcation -> Zero-shot classifcation to predict topic of article.
3. Retrival-augmented classification -> Modify each component of pipeline to optimize performance
4. Routing for Inference -> Pipeline followed by article for each compoenent depend upon cluster/type.
5. Diversity for recommended -> Diversify recommended article based on cluster

Basically, clustering help improve precision for each model.

For our project, we will use clustering for
1. **`Routing Semantic search`**: <br>
    For large search space, searching entire database is super slow [High latency] -> use clustering to focus on relevant articles. <br>
    This will narrow down our semantic search space

2. **`Zero-shot classification`**: <br>
    For cold-start classification, we can use clustering to focus on relevant articles. <br>

3. **`Visualization`**: <br>
   We can use cluster base to visualize our data. [targeted wordcloud etc.]

In [ ]:
# After experimentation - Optimal Clusters = 50, heuristic_weight = 0.3
clus_matrix = prepare_clustering_matrix(Dataset, heuristic_cols=Heuristic_cols, embedding_col='Embedding',
    scaling_method='log-robust', apply_reduction=True, embedding_weight=1.0, heuristic_weight=0., reduction_factor=0.5)


clusterer = SemanticClustering(n_clusters=10, random_state=42)
class_labels = clusterer.fit(clus_matrix)
print(clusterer.evaluate(clus_matrix, class_labels))

clusterer2, class_labels2 = Cluster_articles_hdbscan(clus_matrix)
print(evaluate_hdbscan(clus_matrix, class_labels2))

print(pd.Series(class_labels).value_counts())
print(pd.Series(class_labels2).value_counts())

### Experimentation:

In [ ]:
# import numpy as np
# import pandas as pd
# from itertools import product
# from tqdm import tqdm

# def grid_search_clustering(
#     Dataset,
#     Heuristic_cols,
#     embedding_col='Embedding',
#     scaling_method='log-robust',
#     apply_reduction=True,
#     n_clusters=10,
#     random_state=42
# ):
#     # Define parameter grid
#     embedding_weights = [0.0, 0.25 ,0.5, 0.75, 1.0, 1.5, 2.0]
#     heuristic_weights = [0.0, 0.1, 0.3, 0.5]
#     reduction_factors = [0.1, 0.3, 0.5]

#     results = []

#     combinations = list(product(embedding_weights, heuristic_weights, reduction_factors))

#     for emb_w, heur_w, red_f in tqdm(combinations):
#         # Optional constraint: avoid meaningless combinations
#         if emb_w == 0 and heur_w == 0:
#             continue

#         try:
#             # Prepare matrix
#             clus_matrix = prepare_clustering_matrix(
#                 Dataset,
#                 heuristic_cols=Heuristic_cols,
#                 embedding_col=embedding_col,
#                 scaling_method=scaling_method,
#                 apply_reduction=apply_reduction,
#                 embedding_weight=emb_w,
#                 heuristic_weight=heur_w,
#                 reduction_factor=red_f
#             )

#             # Cluster
#             clusterer = SemanticClustering(
#                 n_clusters=n_clusters,
#                 random_state=random_state
#             )

#             labels = clusterer.fit(clus_matrix)

#             # Evaluate (silhouette)
#             score = clusterer.evaluate(clus_matrix, labels)


#             results.append({
#                 'embedding_weight': emb_w,
#                 'heuristic_weight': heur_w,
#                 'reduction_factor': red_f,
#                 'silhouette_score': score
#             })

#         except Exception as e:
#             print(f"Failed for {emb_w, heur_w, red_f}: {e}")

#     results_df = pd.DataFrame(results)

#     # Sort by best silhouette score
#     results_df = results_df.sort_values(
#         by='silhouette_score',
#         ascending=False
#     ).reset_index(drop=True)

#     return results_df

# results_df = grid_search_clustering(Dataset, Heuristic_cols)

# print("Top 10 Results:")
# print(results_df.head(10))

# best_params = results_df.iloc[0]
# print("\nBest Parameters:")
# print(best_params)


In [ ]:
# # Re-run with best parameters
# clus_matrix = prepare_clustering_matrix(
#     Dataset,
#     heuristic_cols=Heuristic_cols,
#     embedding_col='Embedding',
#     scaling_method='log-robust',
#     apply_reduction=True,
#     embedding_weight=best_params['embedding_weight'],
#     heuristic_weight=best_params['heuristic_weight'],
#     reduction_factor=best_params['reduction_factor']
# )

# clusterer = SemanticClustering(
#     n_clusters=10,
#     random_state=42
# )

# class_labels = clusterer.fit(clus_matrix)
# print(clusterer.evaluate(clus_matrix, class_labels))

Experiements findings:
- HDBSCAN works only when we have umap dimensionality reduction - > still worse then kmeans manually
- 50 Clusters give best cluster for K-Means
- Mixing heuristic features with embedding is poisoning clusters result. Just use embedding only.
- My TF-IDF over KMeans clusters is just a weak version of BERTopic
- Manual CLuster matrix prep + Clustering mess + TF-IDF topic pull is a huge mess. <br>
- Replace entire pipeline with BERTopic.

In [ ]:
# Adding label to articles and saving model
Dataset["Cluster"] = class_labels
Dataset.to_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")

# Saving models
joblib.dump(clusterer, '../Models/clusterer.joblib')

### **Zero-Shot Cluster labeling**:
We will use the clusters created & label those groups based on Keywords present in them.

In [ ]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Load spaCy once
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


INVALID_TERMS = {
    "mr", "mrs", "ms", "dr",
    "said", "say", "says", "saying",
    "told", "according"
}


def is_valid_topic_term(term: str) -> bool:
    """
    Linguistic filter for topic terms
    """
    if not term.isalpha():
        return False
    if term.lower() in INVALID_TERMS:
        return False
    if len(term) <= 2:
        return False

    doc = nlp(term)
    for token in doc:
        # reject verbs, auxiliaries, pronouns
        if token.pos_ in {"VERB", "AUX", "PRON"}:
            return False

    return True


def label_cluster_df(
    df: pd.DataFrame,
    text_col: str = "Content",
    cluster_col: str = "Cluster",
    top_n_words: int = 10,
    min_df: int = 5,
    max_df: float = 0.75,
    candidate_pool: int = 40
):
    """
    Label KMeans clusters using TF-IDF + linguistic filtering.
    """

    cluster_labels = {}

    for cluster_id in sorted(df[cluster_col].unique()):
        if cluster_id == -1:
            continue

        texts = df.loc[df[cluster_col] == cluster_id, text_col].astype(str)

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=min_df,
            max_df=max_df
        )

        tfidf = vectorizer.fit_transform(texts)
        avg_scores = np.asarray(tfidf.mean(axis=0)).ravel()
        vocab = vectorizer.get_feature_names_out()

        # Take more candidates than needed, then filter linguistically
        sorted_terms = vocab[np.argsort(avg_scores)[::-1]]
        filtered_terms = []

        for term in sorted_terms[:candidate_pool]:
            if is_valid_topic_term(term):
                filtered_terms.append(term)
            if len(filtered_terms) == top_n_words:
                break

        cluster_labels[cluster_id] = filtered_terms

    return cluster_labels

cluster_labels = label_cluster_df(Dataset, text_col="Content", cluster_col="Cluster", top_n_words=10, min_df=5, max_df=0.75, candidate_pool=20)
print(cluster_labels)

In [ ]:
# Saving labels
joblib.dump(cluster_labels, '../Models/Components/manual_cluster_labels.joblib')

This is valid but not strong enough to get meaniful cluster labels. <br>
We will improve it using BERTopic.

### **BERTopic Pipeline**:

> Instead of add onto our batch K-means clustering, we will pivot into a complete Bertopic modeling pipeline which will
1. Fit bertopic to create cluster
2. Label cluster using zero-shot learning
3. Use these cluster for better visualization and analysis at inference time.
4. Then from these 

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
## BERTopic Zero-shot Cluster Labeling
from bertopic import BERTopic

# 1. Embedding
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = np.vstack(Dataset['Embedding'].tolist())
# 2. Control dimensionality
umap_model = umap.UMAP(n_components=5, n_neighbors=30, min_dist=0.0, metric='cosine', random_state=42)
# 3. Handle the NOISE
cluster_model = MiniBatchKMeans(n_clusters=20, random_state=42)
# hdbscan_model = hdbscan.HDBSCAN(min_samples=5, min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# 4. Cluster topics
vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english', min_df=10, max_df=0.8)


# 5. Main pipeline model
topic_model = BERTopic(language='english',vectorizer_model=vectorizer, 
    embedding_model=embedding_model, umap_model=umap_model, hdbscan_model=cluster_model, 
    calculate_probabilities=False, verbose=True, nr_topics=None # let kmeans handle the count
)
topics, probs = topic_model.fit_transform(Dataset['Content'].tolist(), embeddings)

Analysing Bert findings

In [ ]:
# 6. Topic Analytics
Dataset['Topic'] = topics
Dataset.to_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")

agg_map = {
    'Content': 'count',                             # Volume of articles
    # 'char_count': ['mean','std'], 'sentence_count': ['mean', 'std'], 'word_count': ['mean', 'std'], 'unique_word_count': ['mean', 'std'],
    'word_ratio': 'mean', 'char_ratio': 'mean', 'sent_ratio': 'mean',
    'lexical_diversity': 'mean',
    'stopword_ratio': 'mean',
    'noun_count': 'mean', 'verb_count': 'mean', 'adj_count': 'mean', 'adv_count': 'mean', 'pronoun_count': 'mean',
    'person_count': 'mean', 'gpe_count': 'mean', 'org_count': 'mean',
    'flesch_reading_ease': 'mean', 'flesch_kincaid_grade': 'mean'
}
Dataset.groupby('Topic').agg(agg_map)

Manual inspection of topics

In [ ]:
topics = topic_model.get_topics()            # Each topic key words
cluster_sizes = topic_model.topic_sizes_        # Number of articles in each topic
topic_labels = topic_model.topic_labels_        # Each topic label/title
topic_embedding = topic_model.topic_embeddings_ # Each topic numerical representation
topic_info = topic_model.get_topic_info().set_index('Topic')

display("Topic information:", topic_info)

All useful plots

In [ ]:
# How topics relate to each other
visual_dist = topic_model.visualize_topics()  # Locate nearest cluster
# Shows correlation matrix b/we topics based on their word representations
cluster_similarity_score = topic_model.visualize_heatmap()  # Get nearest clusters
# Dendogram showing how topics cluster together -> Merge topics
hierarchy = topic_model.visualize_hierarchy()  # Hierarchical view of topics

In [ ]:
# saving entire model:
topic_model.save("../Models/TopicModel", serialization="pytorch", save_embedding_model=True)

In [ ]:
joblib.dump(topic_model, '../Models/topic_model.joblib') # Save topic_model

We will use this independent topic modeling for analysis and visualization at infernce.


> NOTE: This is a simple project where we are using pre-trained model to focus on application part at inference time. <br>
All of these focused improvisations can be done at larger scale or at later expansions.

For now, we mostly just follow our normal pipeline.

#### 3. Semantic Search:
We will cluster our article based on their embedding and then use this cluster to narrow down our search space for semantic search. <br>
This will improve our search latency and precision. <br>

In [ ]:
from bertopic import BERTopic
import joblib
import pandas as pd

df = pd.read_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")

# loading model
bert_joblib = joblib.load('../Models/topic_model.joblib')
topic_model = BERTopic.load('../Models/TopicModel', embedding_model='all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1508.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
bert_joblib.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,1032,0_mps_tory_nhs_referendum,"[mps, tory, nhs, referendum, mr cameron, clima...","[By . Tim Shipman . PUBLISHED: . 16:30 EST, 28..."
1,1,859,1_chelsea_liverpool_arsenal_champions,"[chelsea, liverpool, arsenal, champions, world...",[Chelsea return to Anfield on Saturday seven m...
2,2,787,2_attorney_prosecutors_sheriff_nfl,"[attorney, prosecutors, sheriff, nfl, convicte...",[By . Associated Press Reporter . PUBLISHED: ....
3,3,758,3_syria_iran_iraq_islamic,"[syria, iran, iraq, islamic, qaeda, al qaeda, ...","[U.S. airstrikes ""are not going to save"" the k..."
4,4,603,4_court heard_powell_custody_disappearance,"[court heard, powell, custody, disappearance, ...","[Rebecca Durkin, 19, will face trial for the m..."
5,5,594,5_jailed_pope_offences_court heard,"[jailed, pope, offences, court heard, vatican,...","[By . Steve Nolan . PUBLISHED: . 14:56 EST, 3 ..."
6,6,536,6_album_actress_comedy_novel,"[album, actress, comedy, novel, musical, petty...",[Winning an Academy Award is the pinnacle of s...
7,7,535,7_bedroom_disney_auction_coins,"[bedroom, disney, auction, coins, maps, graffi...","[Hostels were once the crowded, dirty havens f..."
8,8,434,8_cricket_rugby_wins_rangers,"[cricket, rugby, wins, rangers, second half, c...","[Jordan Obita scored the decisive spot-kick, t..."
9,9,417,9_trump_republican_republicans_romney,"[trump, republican, republicans, romney, democ...",[(CNN) -- If Republicans regain control of the...


In [7]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,1032,0_mps_tory_nhs_referendum,"[mps, tory, nhs, referendum, mr cameron, clima...",NaN
1,1,859,1_chelsea_liverpool_arsenal_champions,"[chelsea, liverpool, arsenal, champions, world...",NaN
2,2,787,2_attorney_prosecutors_sheriff_nfl,"[attorney, prosecutors, sheriff, nfl, convicte...",NaN
3,3,758,3_syria_iran_iraq_islamic,"[syria, iran, iraq, islamic, qaeda, al qaeda, ...",NaN
4,4,603,4_court heard_powell_custody_disappearance,"[court heard, powell, custody, disappearance, ...",NaN
5,5,594,5_jailed_pope_offences_court heard,"[jailed, pope, offences, court heard, vatican,...",NaN
6,6,536,6_album_actress_comedy_novel,"[album, actress, comedy, novel, musical, petty...",NaN
7,7,535,7_bedroom_disney_auction_coins,"[bedroom, disney, auction, coins, maps, graffi...",NaN
8,8,434,8_cricket_rugby_wins_rangers,"[cricket, rugby, wins, rangers, second half, c...",NaN
9,9,417,9_trump_republican_republicans_romney,"[trump, republican, republicans, romney, democ...",NaN


In [ ]:
query = df.query('word_count > 100').sample()['Content'].values[0]
# print(topic_model.embedding_model.embed_documents(query))

# print(dir(topic_model.embedding_model))
# print(help(topic_model.embedding_model.word_embedding_model))
print(query)

In [ ]:
bbc = pd.read_parquet("../Dataset/Clean/BBC_with_features.parquet")
test = bbc.query('word_count > 200').sample()['text'].values[0]
test

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def advance_search(query:str|List[str], model:BERTopic, df:pd.DataFrame,
    embeddings:np.ndarray=None, top_k:int=5, filter_space:bool=True):
    """
    Advance semantic search function with Topic Routing
    
    Args:
        query (str | List[str]): Query Article or list of queries
        model (BERTopic): Fitted BERTopic model pipeline
        df (pd.DataFrame): Dataset
        embeddings (np.ndarray): Topic embeddings from BERTopic
        top_k (int, optional): No. of results to return.
        filter_space (bool, optional): Filter search space based on Topic Cluster matching.

    Returns:
        pd.DataFrame: Top K similar articles
    """
    if embeddings is None:
        embeddings = np.vstack(df['Embedding'])
    
    # 1. Embed the query
    query_vec = model.embedding_model.embed_documents([query])
    # 2. Predict topic of query (Zero-Shot Routing)
    pred_topic, prob = model.transform([query])
    
    # 3. Filter search space (Optimization)
    search = df.index
    if filter_space and pred_topic[0] != -1:
        mask = (df['Topic'] == pred_topic[0])
        if mask.sum() > 0:
            search = df[mask].index
        
    if search.shape[0] == 0:
        search = df.index # Falling back to entire dataset
    
    # Filter embeddings
    filter_em = embeddings[search]
    
    # 4. Semantic search
    sim_search = cosine_similarity(query_vec, filter_em)
    # 5. Get Top K
    best_local_idx = np.argsort(sim_search)[0][::-1][:top_k]
    # print(np.argsort(sim_search), best_local_idx)
    
    # similar_embeddings =  filter_em[best_local_idx]
    return df.loc[best_local_idx]


In [ ]:
result = advance_search(test, topic_model, df, top_k=5, filter_space=False)
print(test)
print("-"*50)
print(result['Content'].values[4])

Batches: 100%|██████████| 1/1 [00:00<00:00, 64.79it/s]

2026-02-13 01:03:05,077 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
<p>
uk economy ends year with spurt the uk economy grew by an estimated 3.1% in 2004 after accelerating in the last quarter of the year says the office for national statistics (ons). the figure is in line with treasury and bank of england forecasts. the ons says gross domestic product (gdp) rose by a strong 0.7% in the three months to 31 december compared with 0.5% in the previous quarter. the rise came despite a further decline in production output and the worst christmas for retailers in decades. the annual figure marked out the best year since 2000 and was also well ahead of the 2.2% recorded in 2003. growth in the final three months of 2004 marked the 50th consecutive quarter of expansion. on the basis of the latest information the uk has entered 2005 on course to continue its record period of growth said paul boateng chief secretary to the treasury in a statement. the ons said the services sector which accounts for nearly three-quarters of the uk economy grew 1.0% in the quarter. the strong services figure was welcomed by analysts given lacklustre retail sales in december and across the christmas holiday period. the fact that other services components are doing so well suggests to me that we are back to trend (growth) and i am not particularly concerned about any further slowdown said ross walker uk economist at rbs financial markets. however output in the production sector contracted 0.5% the second quarterly fall in row and a state of affairs that some economists classify as a recession. however the ons would not comment on the definition of a recession and whether the manufacturing recovery was over. but steve radley chief economist at the manufacturers organisation eef said: these figures remain at odds with what is actually happening on the ground. whilst companies may be experiencing tougher conditions this year recession is not a word that manufacturers would currently recognise. the ons said a sharp fall in mining and quarrying which was driven by oil and gas extraction was primarily responsible for the overall contraction in manufacturing production figures. simon rubinsohn chief economist at gerrard said: this outturn (of 0.7%) was well ahead of the market expectations and cast doubt on the scare stories doing the rounds surrounding the current state of the uk economy. and he said the gdp figures may help to push interest rate expectations a little higher along the curve . the suggestion from the money markets is that the next move is now more likely to be in an upward rather than a downward direction. this is consistent with our own thinking said mr rubinsohn. the bank of england s nine-strong rate-setting committee voted unanimously earlier this month to keep interest rates steady at 4.75% minutes of the meeting showed on wednesday.
</p>
-------------------------------------------------- <br>
Despite the increase, manufacturing output was still down 0.6% from the same month last year.
The wider measure of industrial production fell 0.2% in September, but was 1.1% higher than a year earlier.
Other figures from the ONS indicated the goods trade deficit narrowed to £9.35bn in September.
The deficit - showing that the UK imported more goods than it exported - was down from a gap of £10.79bn in August.
The deficit in goods and services narrowed to £1.4bn in September from £2.9bn the previous month.
However, for the July-to-September quarter, the trade deficit in goods and services widened to £8.5bn, from a gap of £5.1bn in the previous quarter.
Last month, the ONS said the UK's economy grew by 0.5% in the third quarter, in its first growth estimate for the quarter.
The latest ONS figures show that industrial production rose by 0.2% in the July-to-September period, slightly below initial estimates of 0.3%, while manufacturing output fell by 0.4%.
Lee Hopley, chief economist at the manufacturers' organisation, the EEF, said: "While manufacturing contracted in the last quarter, there are signs that some parts of industry at least were mounting a comeback after a summer lull."
She added, though, that there were risks from the economic slowdown happening elsewhere around the world: "Another disappointing set of trade figures for manufacturing show that these effects are already being felt, with a significant fall in goods exports to China over the past three months."

As we can see from output, Both article have very similar context -> UK, economy, growth, numbers, etc.

This component has setbacks/concerns. <br>
This is One part of our pipeline which will have huge problem with scaling as here is where we have to interact with training dataset at inference time.

This will cause latency issue at larger scales. So, we will need to optimized this component using `FAISS`.

#### what is FAISS? <br>
`FAISS` (Facebook AI Similarity Search) is an open-source library developed by **Meta AI** for efficient *similarity search* and *clustering* of dense, high-dimensional vectors. <br>
It enables fast retrieval of similar items (e.g., images, text) in large-scale datasets, often exceeding RAM, using advanced indexing techniques and GPU acceleration. 

In [ ]:
import faiss
import logging

# Configure logging for production-grade output
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("SearchEngine")

class SearchEngine:
    def __init__(self, model:BERTopic, df:pd.DataFrame, embeddings:np.ndarray=None):
        """
        Initializes a FAISS Search Engine
        creates seperate vector index for each Topic to maximize speed and relevence
        """
        if embeddings is None:
            embeddings = np.vstack(df['Embedding'].values)
        print(embeddings.shape)
        
        self.topic_model = model
        self.df = df
        
        self.indices = {}   # Dict to store FAISS index per topic
        self.id_maps = {}   # Dict to store mapping from FAISS ID -> Dataframe index
        
        # 1. Normalize Embeddings -> Inner Product == Cosine Similarity
        self.embeddings = self._normalize(embeddings)
        self.em_dim = self.embeddings.shape[1]
        
        self._build_indices(df)
        
    def _normalize(self, v):
        norm = np.linalg.norm(v, axis=1, keepdims=True)
        return (v/norm).astype("float32")
        
    def _build_indices(self, df:pd.DataFrame):
        """
        Build a separate IndexFlatIP for each topic cluster
        """
        unique_topics = df['Topic'].unique()
        
        for topic_id in unique_topics:
            # 1. Boolean mask for topic match
            mask = (df['Topic']==topic_id).values
            # 2. Specific embedding for this cluster
            subset_embedding = self.embeddings[mask]
            # 3. Store the original DataFrame indices to map back
            self.id_maps[topic_id] = df.index[mask].values
            
            # 4. Create FAISS index (Inner product)
            idx = faiss.IndexFlatIP(self.em_dim)    # Inner Product
            """ For larger size datasets, switch to IndexIVFFlat | IndexHNSW """
            idx.add(subset_embedding) # add training set
            
            self.indices[topic_id] = idx
        
        # Adding complete dataset as well
        self.id_maps[-1] = df.index
        idx = faiss.IndexFlatIP(self.em_dim)
        idx.add(self.embeddings)
        self.indices[-1] = idx
            
    def search(self, query:str, top_k:int = 5) -> pd.DataFrame:
        """
        Perform a routed semantic search
        
        return: 
            similar articles info, distance based score, cluster id
        """
        # 1. Embed Query
        query_vec = self.topic_model.embedding_model.embed([query])
        query_vec = self._normalize(query_vec).astype("float32")
        
        # 2. Topic Routing (zero-shot)
        pred_topics, _ = self.topic_model.transform(query)
        target_topic = pred_topics[0]
        
        # 3. Fallback
        if target_topic not in self.indices:
            target_topic = -1 # Complete dataset
            
        # 4. Sharded search
        index = self.indices[target_topic]
        k = min(top_k, index.ntotal) # safety, dont request more then cluster size
        
        # D = Distances(Scores), I=Indices (FAISS)
        D, I = index.search(query_vec, k)
        # print(D[0], I[0])
        
        # 5. Map FAISS ids back to get dataframe
        df_index = self.id_maps[target_topic][I[0]]
        scores = D[0]
        
        return self.df.loc[df_index], scores, target_topic
        

In [ ]:
seo = SearchEngine(topic_model, df)
result,scores,topic = seo.search(test)
print("Distances: ", scores)
print("Query topic: ", topic)
print("Results index: ", result.index)
result